# 02 – Understanding the Data

**Projekt:** WealthScope AI  
**Kontext:** QUA3CK / Big-Data / Machine-Learning / Streamlit-App  
**Datenbasis:** Kaggle Stock/ETF Dataset, lokal verarbeitet  
**Hinweis:** Diese Notebooks dienen der wissenschaftlichen Nachvollziehbarkeit. Sie ersetzen keine Finanzberatung.

## Ziel

Dieses Notebook prüft die Datenbasis:

- Größe
- Spalten
- Zeitraum
- Ticker-Verteilung
- Asset-Typen
- fehlende Werte
- Zielvariable

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

PARQUET_PATH = DATA_DIR / "wealthscope_features.parquet"
CSV_PATH = DATA_DIR / "wealthscope_features.csv"

def load_features():
    if PARQUET_PATH.exists():
        df = pd.read_parquet(PARQUET_PATH)
        source = "REAL_PARQUET"
    elif CSV_PATH.exists():
        df = pd.read_csv(CSV_PATH)
        source = "REAL_CSV"
    else:
        raise FileNotFoundError("Kein Feature-Datensatz gefunden. Erwartet wealthscope_features.parquet oder .csv")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    return df, source

df, source = load_features()

print("Datenquelle:", source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
profile = {
    "rows": len(df),
    "columns": len(df.columns),
    "tickers": df["ticker"].nunique() if "ticker" in df.columns else None,
    "date_min": df["date"].min() if "date" in df.columns else None,
    "date_max": df["date"].max() if "date" in df.columns else None,
    "target_20d_available": "target_20d" in df.columns,
}
pd.DataFrame([profile]).T.rename(columns={0: "Wert"})

In [ ]:
missing = (
    df.isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)
missing.columns = ["Spalte", "Fehlende Werte in %"]
missing.sort_values("Fehlende Werte in %", ascending=False).head(30)

In [ ]:
ticker_counts = df["ticker"].value_counts().reset_index()
ticker_counts.columns = ["ticker", "rows"]
ticker_counts

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(ticker_counts["ticker"], ticker_counts["rows"])
plt.title("Datenpunkte je Ticker")
plt.xticks(rotation=45)
plt.ylabel("Zeilen")
plt.show()

In [ ]:
if "asset_type" in df.columns:
    asset_counts = df["asset_type"].value_counts()
    display(asset_counts)

    plt.figure(figsize=(6, 4))
    plt.bar(asset_counts.index.astype(str), asset_counts.values)
    plt.title("Asset-Typen")
    plt.ylabel("Zeilen")
    plt.show()

## Interpretation

Diese Analyse belegt, dass die App auf einer echten Datenbasis arbeitet und nicht auf einer Demo-Tabelle.